## Blockchain Governance Capability (BGC)

In [3]:
# ================================================================
# BGC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT BGC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "BGC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "BGC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No BGC responses were found.

        Check that BGC appears in the participant sheets.
        """
    )

print("\nBGC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "BGC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nBGC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW BGC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("BGC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. BGC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved BGC mappings here after reviewing
# the actual BGC raw themes.
#
# Example:
#
# "shared governance":
#     "Collaborative blockchain governance",
#
# "joint governance":
#     "Collaborative blockchain governance",
#
# ------------------------------------------------

BGC_NORMALIZATION = {

    # ADD BGC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in BGC_NORMALIZATION:

        normalized = BGC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × BGC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "BGC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "BGC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("BGC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique BGC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized BGC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

BGC original responses:
26

BGC cleaned theme observations:
100


BGC RAW THEMES
                    Raw_Theme                   Theme_Clean                     Theme_Key
                       Access                        Access                        access
                       access                        access                        access
                access rights                 access rights                 access rights
               accountability                accountability                accountability
               Accountability                Accountability                accountability
                   compliance                    compliance                    compliance
        com

In [9]:
# ================================================================
# COMPLETE BGC QUALITATIVE CODING ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("BGC_Coding_20260825_132218.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as "
        "your Python notebook."
    )

print("=" * 70)
print("BGC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    # Exact match
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible match
    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_BGC",
    "01_Original",
    "Original_BGC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_BGC",
    "02_Raw",
    "Raw_BGC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_BGC",
    "03_Cleaned",
    "Cleaned_BGC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the BGC Coding/Normalized Coding sheet."
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    # Exact match
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    # Flexible match
    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


# Rename to standard names

coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned BGC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned BGC sheet."
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# P01–P26 ordering

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY
# ================================================================

participant_columns = existing + other

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE
# ================================================================

total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 18. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 19. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_BGC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 25. BGC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "BGC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL BGC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_BGC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 28. SAVE COMPLETE BGC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"BGC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Prevent accidental overwrite

counter = 1

base_output = output_file

while output_file.exists():

    output_file = Path(
        f"BGC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_BGC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("BGC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized BGC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL BGC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL BGC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_BGC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete BGC qualitative analysis workbook "
    "created successfully."
)

BGC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\BGC_Coding_20260825_132218.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


BGC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations

In [ ]:
# ================================================================
# BGC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# BGC_FINAL_QUALITATIVE_ANALYSIS_20260825_144139.xlsx
#
# OUTPUT:
# BGC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_144329.xlsx
#
# PURPOSE:
# 1. Preserve the completed BGC qualitative coding
# 2. Extract final BGC evidence
# 3. Organize final themes into evidence-based BGC dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring review
# 6. Create dimension × theme evidence tables
#
# NOTE:
# These are QUALITATIVE dimensions derived from expert evidence.
# They are NOT yet statistically validated subdimensions.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "BGC_FINAL_QUALITATIVE_ANALYSIS_20260825_144139"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:
    raise FileNotFoundError(
        "\nBGC input file was not found.\n\n"
        f"Expected file:\n{TARGET}.xlsx\n\n"
        "Put the Excel file in the same folder as your notebook."
    )

INPUT_FILE = possible_files[0]

print("=" * 80)
print("BGC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:
    try:
        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )
    except Exception as e:
        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_BGC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_BGC_Evidence was not found."
    )


print(
    "\nFinal BGC evidence sheet:",
    final_sheet
)


# ================================================================
# 6. READ FINAL BGC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY THEME COLUMN
# ================================================================

theme_col = None

for c in final_evidence.columns:

    if str(c).strip() == "Final_BGC_Theme":

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if (
            "BGC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify Final BGC Theme column."
    )


final_evidence[
    "Final_BGC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE PREVALENCE COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_BGC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


# Your study has 26 experts.
# The participant matrix is used when available.

if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

sort_columns = []

if "Experts_Mentioning" in final_evidence.columns:

    sort_columns.append(
        "Experts_Mentioning"
    )

sort_columns.append(
    "Final_BGC_Theme"
)


final_evidence = (
    final_evidence
    .sort_values(
        sort_columns,
        ascending=[
            False
            if c == "Experts_Mentioning"
            else True
            for c in sort_columns
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# 13. FINAL BGC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_BGC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_bgc_evidence = final_evidence[
    evidence_columns
].copy()


# Prevent Rank duplication

if "Rank" in final_bgc_evidence.columns:

    final_bgc_evidence = (
        final_bgc_evidence
        .drop(columns=["Rank"])
    )


final_bgc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_bgc_evidence) + 1
    )
)


# ================================================================
# 14. BGC DIMENSION MAPPING
# ================================================================
#
# Dimensions are derived from the actual BGC themes in the
# attached workbook.
#
# ================================================================


def map_bgc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. ROLES, RESPONSIBILITIES & ACCOUNTABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "accountability",
            "responsibility",
            "responsibilities",
            "roles",
            "ownership",
            "participant responsibility",
            "participant obligations",
            "partner obligations",
            "network responsibility",
            "data responsibility",
            "governance responsibilities"

        ]
    ):

        return (
            "Roles, Responsibilities & Accountability"
        )


    # ------------------------------------------------------------
    # 2. PARTICIPATION, ACCESS & AUTHORITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "access",
            "access rights",
            "participation",
            "participation rules",
            "participant admission",
            "information authority",
            "contribution authority",
            "data-entry authority",
            "modification rights",
            "contribution"

        ]
    ):

        return (
            "Participation, Access & Authority"
        )


    # ------------------------------------------------------------
    # 3. GOVERNANCE RULES, STANDARDS & COMPLIANCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "compliance",
            "compliance monitoring",
            "standards",
            "data standards",
            "data-quality standards",
            "predefined rules",
            "governance rules",
            "governance",
            "procedures",
            "mandatory information",
            "information requirements",
            "prior agreements"

        ]
    ):

        return (
            "Governance Rules, Standards & Compliance"
        )


    # ------------------------------------------------------------
    # 4. DATA QUALITY, VALIDATION & INFORMATION CONTROL
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "verification",
            "validation",
            "data quality",
            "information quality",
            "inaccurate records",
            "correction",
            "correction procedures",
            "corrective action",
            "error resolution",
            "error handling"

        ]
    ):

        return (
            "Data Quality, Validation & Information Control"
        )


    # ------------------------------------------------------------
    # 5. DISPUTE, EXCEPTION & PROBLEM MANAGEMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "dispute",
            "disputes",
            "dispute resolution",
            "dispute handling",
            "dispute management",
            "disagreement management",
            "disagreement resolution",
            "exception handling",
            "failure procedures",
            "problem resolution"

        ]
    ):

        return (
            "Dispute, Exception & Problem Management"
        )


    # ------------------------------------------------------------
    # 6. INTERORGANIZATIONAL GOVERNANCE ALIGNMENT & TRUST
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "interorganizational agreement",
            "interorganizational trust",
            "governance alignment"

        ]
    ):

        return (
            "Interorganizational Governance Alignment & Trust"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_bgc_evidence[
    "BGC_Dimension"
] = (
    final_bgc_evidence[
        "Final_BGC_Theme"
    ]
    .apply(
        map_bgc_dimension
    )
)


# ================================================================
# 15. EXPLICIT MAPPING FOR AMBIGUOUS THEMES
# ================================================================
#
# This resolves capitalization and short generic labels without
# altering the original qualitative evidence.
# ================================================================

manual_mapping = {

    # Roles / responsibility

    "Accountability":
        "Roles, Responsibilities & Accountability",

    "accountability":
        "Roles, Responsibilities & Accountability",

    "Responsibility":
        "Roles, Responsibilities & Accountability",

    "responsibility":
        "Roles, Responsibilities & Accountability",

    "Responsibilities":
        "Roles, Responsibilities & Accountability",

    "responsibilities":
        "Roles, Responsibilities & Accountability",

    "Roles":
        "Roles, Responsibilities & Accountability",

    "Ownership":
        "Roles, Responsibilities & Accountability",

    "Network responsibility":
        "Roles, Responsibilities & Accountability",

    "Data responsibility":
        "Roles, Responsibilities & Accountability",

    "data responsibility":
        "Roles, Responsibilities & Accountability",

    "Participant responsibility":
        "Roles, Responsibilities & Accountability",

    "Participant obligations":
        "Roles, Responsibilities & Accountability",

    "Partner obligations":
        "Roles, Responsibilities & Accountability",

    "Governance responsibilities":
        "Roles, Responsibilities & Accountability",


    # Participation / access / authority

    "access":
        "Participation, Access & Authority",

    "Access":
        "Participation, Access & Authority",

    "access rights":
        "Participation, Access & Authority",

    "Participation":
        "Participation, Access & Authority",

    "participation":
        "Participation, Access & Authority",

    "Participation rules":
        "Participation, Access & Authority",

    "participant admission":
        "Participation, Access & Authority",

    "Information authority":
        "Participation, Access & Authority",

    "Contribution authority":
        "Participation, Access & Authority",

    "contribution":
        "Participation, Access & Authority",

    "Data-entry authority":
        "Participation, Access & Authority",

    "modification rights":
        "Participation, Access & Authority",


    # Rules / standards / compliance

    "compliance":
        "Governance Rules, Standards & Compliance",

    "compliance monitoring":
        "Governance Rules, Standards & Compliance",

    "standards":
        "Governance Rules, Standards & Compliance",

    "Data standards":
        "Governance Rules, Standards & Compliance",

    "data-quality standards":
        "Governance Rules, Standards & Compliance",

    "Predefined rules":
        "Governance Rules, Standards & Compliance",

    "Governance rules":
        "Governance Rules, Standards & Compliance",

    "Governance":
        "Governance Rules, Standards & Compliance",

    "Procedures":
        "Governance Rules, Standards & Compliance",

    "information requirements":
        "Governance Rules, Standards & Compliance",

    "mandatory information":
        "Governance Rules, Standards & Compliance",

    "prior agreements":
        "Governance Rules, Standards & Compliance",


    # Data quality / validation

    "verification":
        "Data Quality, Validation & Information Control",

    "validation":
        "Data Quality, Validation & Information Control",

    "data quality":
        "Data Quality, Validation & Information Control",

    "information quality":
        "Data Quality, Validation & Information Control",

    "inaccurate records":
        "Data Quality, Validation & Information Control",

    "correction":
        "Data Quality, Validation & Information Control",

    "correction procedures":
        "Data Quality, Validation & Information Control",

    "corrective action":
        "Data Quality, Validation & Information Control",

    "error resolution":
        "Data Quality, Validation & Information Control",

    "error handling":
        "Data Quality, Validation & Information Control",


    # Dispute / exception

    "dispute resolution":
        "Dispute, Exception & Problem Management",

    "disputes":
        "Dispute, Exception & Problem Management",

    "dispute handling":
        "Dispute, Exception & Problem Management",

    "dispute management":
        "Dispute, Exception & Problem Management",

    "Disagreement management":
        "Dispute, Exception & Problem Management",

    "disagreement resolution":
        "Dispute, Exception & Problem Management",

    "exception handling":
        "Dispute, Exception & Problem Management",

    "failure procedures":
        "Dispute, Exception & Problem Management",

    "problem resolution":
        "Dispute, Exception & Problem Management",


    # Interorganizational governance

    "Interorganizational agreement":
        "Interorganizational Governance Alignment & Trust",

    "interorganizational trust":
        "Interorganizational Governance Alignment & Trust",

    "governance alignment":
        "Interorganizational Governance Alignment & Trust"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_bgc_evidence[
            "Final_BGC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_bgc_evidence.loc[
        mask,
        "BGC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_bgc_evidence
    .groupby(
        "BGC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_BGC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate experts mentioning dimension
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        first_col = temp.columns[0]

        temp[
            "_Theme"
        ] = (
            temp[
                first_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [
            p
            for p in participants
            if p in matched.columns
        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback using final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "BGC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_BGC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "BGC_Dimension",

    "Final_BGC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_bgc_evidence.columns
]


dimension_themes = final_bgc_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "BGC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_bgc_evidence[
    final_bgc_evidence[
        "BGC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_BGC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All BGC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [
        c
        for c in coding_audit.columns
        if "Decision" in str(c)
    ]


    if len(decision_columns) > 0:

        decision_col = decision_columns[0]

        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All BGC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final BGC evidence available",

    "Result":
        "PASS"
        if len(final_bgc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_bgc_evidence)} final BGC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "BGC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        (
            len(unmapped_check)
            if "Final_BGC_Theme"
            in unmapped_check.columns
            else 0
        ),

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_bgc_evidence[
                "Final_BGC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"BGC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )

    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    # ------------------------------------------------------------
    # FINAL BGC EVIDENCE
    # ------------------------------------------------------------

    final_bgc_evidence.to_excel(
        writer,
        sheet_name="13_Final_BGC_Evidence",
        index=False
    )

    # ------------------------------------------------------------
    # BGC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_BGC_Dimension_Summary",
        index=False
    )

    # ------------------------------------------------------------
    # BGC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_BGC_Dimension_Themes",
        index=False
    )

    # ------------------------------------------------------------
    # CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("BGC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL BGC EVIDENCE")
print("-" * 80)

print(
    final_bgc_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("BGC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_BGC_Evidence
14_BGC_Dimension_Summary
15_BGC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
For questionnaire development, focus on:

14_BGC_Dimension_Summary
15_BGC_Dimension_Themes

These sheets connect the expert-derived BGC themes
to broader qualitative dimensions.

IMPORTANT:
The dimensions are evidence-based qualitative groupings.
They are NOT yet statistically validated measurement
dimensions. They will be used to develop candidate
questionnaire items.
""")

BGC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\BGC_FINAL_QUALITATIVE_ANALYSIS_20260825_144139.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_BGC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final BGC evidence sheet: 11_Final_BGC_Evidence

Participants used: 26


BGC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\BGC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_144329.xlsx


--------------------------------------------------------------------------------
FINAL BGC EV

## Risk Intelligence Capability (RIC)

In [4]:
# ================================================================
# RIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT RIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RIC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RIC responses were found.

        Check that RIC appears in the participant sheets.
        """
    )

print("\nRIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nRIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW RIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("RIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. RIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved RIC mappings here after reviewing
# the actual RIC raw themes.
#
# Example format:
#
# "risk identification":
#     "Risk identification",
#
# "identifying emerging risks":
#     "Risk identification",
#
# ------------------------------------------------

RIC_NORMALIZATION = {

    # ADD RIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in RIC_NORMALIZATION:

        normalized = RIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × RIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "RIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("RIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RIC original responses:
26

RIC cleaned theme observations:
98


RIC RAW THEMES
                           Raw_Theme                          Theme_Clean                            Theme_Key
                        alternatives                         alternatives                         alternatives
                        anticipation                         anticipation                         anticipation
             Business interpretation              Business interpretation              business interpretation
                  causal connections                   causal connections                   causal connections
                 combined indicators                  combined indicators              

In [ ]:
# ================================================================
# COMPLETE RIC QUALITATIVE CODING ANALYSIS
# Input: RIC_Coding_20260825_132426.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("RIC_Coding_20260825_132426.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as your "
        "Python notebook."
    )

print("=" * 70)
print("RIC QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RIC",
    "01_Original",
    "Original_RIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RIC",
    "02_Raw",
    "Raw_RIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RIC",
    "03_Cleaned",
    "Cleaned_RIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the RIC Coding/Normalized Coding sheet.\n"
        "Available sheets are:\n"
        + "\n".join(
            str(x)
            for x in excel.sheet_names
        )
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned RIC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned RIC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANT LIST
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED RIC THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# ================================================================
# 16. ORDER PARTICIPANTS P01, P02, ...
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


participant_columns = (
    existing + other
)


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 18. PERCENTAGE OF PARTICIPANTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. RIC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_RIC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    coding_audit[
        "Percentage_of_Experts"
    ] = (
        coding_audit[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit[
        "Percentage_of_Experts"
    ] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


if len(coding_df) > 0:

    decision_summary[
        "Percentage"
    ] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary[
        "Percentage"
    ] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_RIC_Themes"
]


# ================================================================
# 26. RIC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "RIC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL RIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_RIC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 29. SAVE COMPLETE RIC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"RIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid overwrite / PermissionError

counter = 1

while output_file.exists():

    output_file = Path(
        f"RIC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RIC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized RIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL RIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT FILE
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete RIC qualitative analysis workbook "
    "created successfully."
)

RIC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\RIC_Coding_20260825_132426.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


RIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations

In [ ]:
# ================================================================
# RIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# RIC_FINAL_QUALITATIVE_ANALYSIS_20260821_101129.xlsx
#
# OUTPUT:
# RIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_144832.xlsx
#
# PURPOSE:
# 1. Preserve the completed RIC qualitative coding
# 2. Extract final RIC evidence
# 3. Organize RIC themes into evidence-based dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring review
# 6. Produce dimension × theme evidence tables
#
# IMPORTANT:
# The dimensions are qualitative groupings for questionnaire
# development. They are NOT statistically validated dimensions.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "RIC_FINAL_QUALITATIVE_ANALYSIS_20260825_144749"

possible_files = []

search_locations = [
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
]

for location in search_locations:

    if location.exists():

        for ext in [".xlsx", ".xlsm", ".xls"]:

            try:
                possible_files.extend(
                    location.rglob(TARGET + ext)
                )
            except Exception:
                pass


# Remove duplicates
possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


if len(possible_files) == 0:

    raise FileNotFoundError(
        "\nRIC input file was not found.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Put the Excel file in the same folder as your "
        "Python notebook or change TARGET to the exact "
        "filename."
    )


INPUT_FILE = possible_files[0]


print("=" * 80)
print("RIC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):

        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:

        x = float(x)

    except:

        return "Not available"

    if x >= 75:

        return "Very High"

    elif x >= 50:

        return "High"

    elif x >= 25:

        return "Moderate"

    else:

        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_RIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_RIC_Evidence was not found."
    )


print(
    "\nFinal RIC evidence sheet:",
    final_sheet
)


# ================================================================
# 6. READ FINAL RIC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. IDENTIFY RIC THEME COLUMN
# ================================================================

theme_col = None


if "Final_RIC_Theme" in final_evidence.columns:

    theme_col = "Final_RIC_Theme"


else:

    for c in final_evidence.columns:

        if (
            "RIC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the Final RIC Theme column."
    )


final_evidence[
    "Final_RIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE IMPORTANT COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_RIC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []


if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    # Study participant count
    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if (
    "Experts_Mentioning"
    in final_evidence.columns
):

    final_evidence = (
        final_evidence
        .sort_values(
            "Experts_Mentioning",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL RIC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_RIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [

    c
    for c in preferred_columns
    if c in final_evidence.columns

]


final_ric_evidence = final_evidence[
    evidence_columns
].copy()


# Remove existing Rank if present
if "Rank" in final_ric_evidence.columns:

    final_ric_evidence = (
        final_ric_evidence
        .drop(columns=["Rank"])
    )


final_ric_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_ric_evidence) + 1
    )
)


# ================================================================
# 14. RIC DIMENSION MAPPING
# ================================================================
#
# These dimensions are derived from the 61 RIC themes present
# in the attached workbook.
#
# ================================================================


def map_ric_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. RISK DETECTION & EARLY IDENTIFICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "early detection",
            "early identification",
            "warning signs",
            "external warnings",
            "connecting signals",
            "signal combination",
            "signal connection",
            "monitoring"

        ]
    ):

        return (
            "Risk Detection & Early Identification"
        )


    # ------------------------------------------------------------
    # 2. RISK EXPOSURE & VULNERABILITY ASSESSMENT
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "exposure",
            "vulnerability",
            "overall exposure",
            "systemic exposure",
            "internal exposure",
            "exposure assessment",
            "contextual exposure",
            "inventory vulnerability",
            "internal vulnerabilities",
            "likelihood",
            "probability",
            "severity",
            "significance"

        ]
    ):

        return (
            "Risk Exposure & Vulnerability Assessment"
        )


    # ------------------------------------------------------------
    # 3. CONTEXTUAL RISK INTERPRETATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "contextual assessment",
            "contextual interpretation",
            "contextual risk interpretation",
            "exposure interpretation",
            "business interpretation",
            "event interpretation",
            "interpretation",
            "conditions",
            "historical patterns",
            "external developments",
            "risk development"

        ]
    ):

        return (
            "Contextual Risk Interpretation"
        )


    # ------------------------------------------------------------
    # 4. RISK INTERCONNECTEDNESS & SYSTEMIC UNDERSTANDING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "dependencies",
            "dependency",
            "interactions",
            "relationships",
            "knock-on effects",
            "event interactions",
            "causal connections",
            "interconnected",
            "reinforcing risks",
            "combined risks",
            "exposure combinations",
            "combined indicators"

        ]
    ):

        return (
            "Risk Interconnectedness & Systemic Understanding"
        )


    # ------------------------------------------------------------
    # 5. RISK AGGREGATION & PRIORITIZATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "combined-risk assessment",
            "risk aggregation",
            "prioritization",
            "ranking",
            "resource focus",
            "alternatives",
            "multiple consequences",
            "consequences",
            "impact"

        ]
    ):

        return (
            "Risk Aggregation & Prioritization"
        )


    # ------------------------------------------------------------
    # 6. RISK INFORMATION INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integrated assessment",
            "inventory",
            "critical materials",
            "external developments",
            "internal vulnerabilities",
            "combined indicators"

        ]
    ):

        return (
            "Risk Information Integration"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_ric_evidence[
    "RIC_Dimension"
] = (
    final_ric_evidence[
        "Final_RIC_Theme"
    ]
    .apply(
        map_ric_dimension
    )
)


# ================================================================
# 15. EXPLICIT MAPPING FOR AMBIGUOUS / DUPLICATE LABELS
# ================================================================

manual_mapping = {

    # ------------------------------------------------------------
    # Detection
    # ------------------------------------------------------------

    "early detection":
        "Risk Detection & Early Identification",

    "early identification":
        "Risk Detection & Early Identification",

    "warning signs":
        "Risk Detection & Early Identification",

    "External warnings":
        "Risk Detection & Early Identification",

    "Monitoring":
        "Risk Detection & Early Identification",

    "Connecting signals":
        "Risk Detection & Early Identification",

    "Signal combination":
        "Risk Detection & Early Identification",

    "Signal connection":
        "Risk Detection & Early Identification",


    # ------------------------------------------------------------
    # Exposure / vulnerability
    # ------------------------------------------------------------

    "Exposure":
        "Risk Exposure & Vulnerability Assessment",

    "exposure":
        "Risk Exposure & Vulnerability Assessment",

    "Exposure interpretation":
        "Contextual Risk Interpretation",

    "Contextual exposure":
        "Risk Exposure & Vulnerability Assessment",

    "Systemic exposure":
        "Risk Exposure & Vulnerability Assessment",

    "internal exposure":
        "Risk Exposure & Vulnerability Assessment",

    "exposure assessment":
        "Risk Exposure & Vulnerability Assessment",

    "inventory vulnerability":
        "Risk Exposure & Vulnerability Assessment",

    "internal vulnerabilities":
        "Risk Exposure & Vulnerability Assessment",

    "vulnerability":
        "Risk Exposure & Vulnerability Assessment",

    "vulnerabilities":
        "Risk Exposure & Vulnerability Assessment",

    "likelihood":
        "Risk Exposure & Vulnerability Assessment",

    "Likelihood":
        "Risk Exposure & Vulnerability Assessment",

    "Probability":
        "Risk Exposure & Vulnerability Assessment",

    "severity":
        "Risk Exposure & Vulnerability Assessment",

    "significance":
        "Risk Exposure & Vulnerability Assessment",

    "Overall exposure":
        "Risk Exposure & Vulnerability Assessment",


    # ------------------------------------------------------------
    # Contextual interpretation
    # ------------------------------------------------------------

    "Contextual assessment":
        "Contextual Risk Interpretation",

    "contextual assessment":
        "Contextual Risk Interpretation",

    "Contextual interpretation":
        "Contextual Risk Interpretation",

    "contextual interpretation":
        "Contextual Risk Interpretation",

    "Contextual risk interpretation":
        "Contextual Risk Interpretation",

    "Business interpretation":
        "Contextual Risk Interpretation",

    "Event interpretation":
        "Contextual Risk Interpretation",

    "interpretation":
        "Contextual Risk Interpretation",

    "conditions":
        "Contextual Risk Interpretation",

    "historical patterns":
        "Contextual Risk Interpretation",

    "Risk development":
        "Contextual Risk Interpretation",


    # ------------------------------------------------------------
    # Interconnectedness
    # ------------------------------------------------------------

    "dependencies":
        "Risk Interconnectedness & Systemic Understanding",

    "dependency":
        "Risk Interconnectedness & Systemic Understanding",

    "interactions":
        "Risk Interconnectedness & Systemic Understanding",

    "relationships":
        "Risk Interconnectedness & Systemic Understanding",

    "knock-on effects":
        "Risk Interconnectedness & Systemic Understanding",

    "Event interactions":
        "Risk Interconnectedness & Systemic Understanding",

    "causal connections":
        "Risk Interconnectedness & Systemic Understanding",

    "Interconnected and reinforcing risks":
        "Risk Interconnectedness & Systemic Understanding",

    "combined risks":
        "Risk Interconnectedness & Systemic Understanding",

    "exposure combinations":
        "Risk Interconnectedness & Systemic Understanding",

    "combined indicators":
        "Risk Interconnectedness & Systemic Understanding",


    # ------------------------------------------------------------
    # Aggregation / prioritization
    # ------------------------------------------------------------

    "prioritization":
        "Risk Aggregation & Prioritization",

    "ranking":
        "Risk Aggregation & Prioritization",

    "resource focus":
        "Risk Aggregation & Prioritization",

    "alternatives":
        "Risk Aggregation & Prioritization",

    "Consequences":
        "Risk Aggregation & Prioritization",

    "consequences":
        "Risk Aggregation & Prioritization",

    "impact":
        "Risk Aggregation & Prioritization",

    "multiple consequences":
        "Risk Aggregation & Prioritization",

    "risk aggregation":
        "Risk Aggregation & Prioritization",

    "combined-risk assessment":
        "Risk Aggregation & Prioritization",


    # ------------------------------------------------------------
    # Information integration
    # ------------------------------------------------------------

    "integrated assessment":
        "Risk Information Integration",

    "inventory":
        "Risk Information Integration",

    "critical materials":
        "Risk Information Integration",

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_ric_evidence[
            "Final_RIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_ric_evidence.loc[
        mask,
        "RIC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_ric_evidence
    .groupby(
        "RIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_RIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate expert prevalence
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        # Identify theme column
        theme_candidates = [
            c
            for c in temp.columns
            if "theme" in str(c).lower()
        ]

        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback to final evidence
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "RIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_RIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSION SUMMARY
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "RIC_Dimension",

    "Final_RIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_ric_evidence.columns
]


dimension_themes = final_ric_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "RIC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_ric_evidence[
    final_ric_evidence[
        "RIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_RIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )


else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RIC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c

        for c in coding_audit.columns

        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All RIC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })


else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final RIC evidence available",

    "Result":
        "PASS"
        if len(final_ric_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_ric_evidence)} final RIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "RIC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_RIC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_ric_evidence[
                "Final_RIC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"RIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # FINAL RIC EVIDENCE
    # ------------------------------------------------------------

    final_ric_evidence.to_excel(
        writer,
        sheet_name="13_Final_RIC_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # RIC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_RIC_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # RIC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_RIC_Dimension_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("RIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL RIC EVIDENCE")
print("-" * 80)

print(
    final_ric_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("RIC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_RIC_Evidence
14_RIC_Dimension_Summary
15_RIC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
For questionnaire development, the two important sheets are:

14_RIC_Dimension_Summary
15_RIC_Dimension_Themes

These provide the bridge from:

Expert responses
      ↓
Raw themes
      ↓
Normalized themes
      ↓
Final RIC themes
      ↓
RIC qualitative dimensions
      ↓
Candidate questionnaire items

The dimensions are qualitative evidence-based groupings.
They are NOT yet statistically validated measurement
dimensions.
""")

RIC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\RIC_FINAL_QUALITATIVE_ANALYSIS_20260825_144749.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final RIC evidence sheet: 11_Final_RIC_Evidence

Participants used: 26


RIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\RIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_144832.xlsx


--------------------------------------------------------------------------------
FINAL RIC EV

## Risk Orchestration Capability (ROC)

In [5]:
# ================================================================
# ROC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT ROC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "ROC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "ROC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No ROC responses were found.

        Check that ROC appears in the participant sheets.
        """
    )

print("\nROC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "ROC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)

print("\nROC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW ROC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("ROC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. ROC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add researcher-approved ROC mappings here after reviewing
# the actual ROC raw themes.
#
# Example format:
#
# "coordinated risk response":
#     "Coordinated risk response",
#
# "risk response coordination":
#     "Coordinated risk response",
#
# ------------------------------------------------

ROC_NORMALIZATION = {

    # ADD ROC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in ROC_NORMALIZATION:

        normalized = ROC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × ROC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "ROC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "ROC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("ROC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique ROC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized ROC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

ROC original responses:
26

ROC cleaned theme observations:
89


ROC RAW THEMES
                               Raw_Theme                              Theme_Clean                                Theme_Key
                              adaptation                               adaptation                               adaptation
                     Adaptive allocation                      Adaptive allocation                      adaptive allocation
                              adjustment                               adjustment                               adjustment
                               alignment                                alignment                                alignment
                           

In [ ]:
# ================================================================
# COMPLETE ROC QUALITATIVE CODING ANALYSIS
# Input: ROC_Coding_20260825_132611.xlsx
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("ROC_Coding_20260825_132611.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"\nFile not found:\n{input_file.resolve()}\n\n"
        "Make sure the Excel file is in the same folder as your "
        "Python notebook."
    )

print("=" * 70)
print("ROC QUALITATIVE CODING ANALYSIS")
print("=" * 70)
print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    for sheet in excel.sheet_names:

        a = (
            str(sheet)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_ROC",
    "01_Original",
    "Original_ROC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_ROC",
    "02_Raw",
    "Raw_ROC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_ROC",
    "03_Cleaned",
    "Cleaned_ROC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
    if original_sheet
    else pd.DataFrame()
)

raw_df = (
    pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
    if raw_sheet
    else pd.DataFrame()
)

clean_df = (
    pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
    if clean_sheet
    else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the ROC Coding/Normalized Coding sheet.\n"
        "Available sheets are:\n"
        + "\n".join(
            str(x)
            for x in excel.sheet_names
        )
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. FIND PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:

    raise ValueError(
        "\nParticipant column not found in Cleaned ROC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. FIND THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key not found in Cleaned ROC sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        [
            "nan",
            "None",
            "",
            "NaN"
        ],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANT LIST
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .replace("", np.nan)
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED DATA WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED ROC THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# ================================================================
# 16. ORDER PARTICIPANTS
# ================================================================

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()

participant_columns = (
    existing + other
)


# ================================================================
# 17. FREQUENCY
# ================================================================

matrix["Frequency"] = (
    matrix[
        participant_columns
    ]
    .sum(axis=1)
)


# ================================================================
# 18. PERCENTAGE OF PARTICIPANTS
# ================================================================

total_participants = len(participants)

if total_participants > 0:

    matrix["Percentage"] = (
        matrix["Frequency"]
        / total_participants
        * 100
    ).round(1)

else:

    matrix["Percentage"] = 0


# ================================================================
# 19. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 20. ROC THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_ROC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 22. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    coding_audit[
        "Percentage_of_Experts"
    ] = (
        coding_audit[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    coding_audit[
        "Percentage_of_Experts"
    ] = 0


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 23. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


if len(coding_df) > 0:

    decision_summary[
        "Percentage"
    ] = (
        decision_summary[
            "Number_of_Raw_Themes"
        ]
        / len(coding_df)
        * 100
    ).round(1)

else:

    decision_summary[
        "Percentage"
    ] = 0


# ================================================================
# 24. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


if total_participants > 0:

    normalization_summary[
        "Percentage_of_Experts"
    ] = (
        normalization_summary[
            "Experts_Mentioning"
        ]
        / total_participants
        * 100
    ).round(1)

else:

    normalization_summary[
        "Percentage_of_Experts"
    ] = 0


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_ROC_Themes"
]


# ================================================================
# 26. ROC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "ROC"
    ],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 27. FINAL ROC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_ROC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ]
    .apply(
        prevalence_category
    )
)


# ================================================================
# 28. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",
        "Original response rows",
        "Raw theme observations",
        "Unique raw themes",
        "Final normalized themes",
        "Unmapped themes",
        "Duplicate participant-theme records",
        "Keep decisions",
        "Merge decisions"

    ],

    "Result": [

        total_participants,
        len(original_df),
        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ] == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ] == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",
        "PASS",
        "PASS"

    ]
})


# ================================================================
# 29. SAVE COMPLETE ROC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"ROC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid overwrite / PermissionError

counter = 1

while output_file.exists():

    output_file = Path(
        f"ROC_FINAL_QUALITATIVE_ANALYSIS_"
        f"{timestamp}_{counter}.xlsx"
    )

    counter += 1


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_ROC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("ROC ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print("Original responses:", len(original_df))

print("Raw theme observations:", len(raw_df))

print("Cleaned theme observations:", len(clean_df))

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized ROC themes:",
    coding_df["Normalized_Theme"].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 31. DISPLAY FINAL ROC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL ROC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_ROC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 32. OUTPUT FILE
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete ROC qualitative analysis workbook "
    "created successfully."
)

ROC QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\ROC_Coding_20260825_132611.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


ROC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations

In [ ]:
# ================================================================
# ROC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# ROC_FINAL_QUALITATIVE_ANALYSIS_20260825_145531.xlsx
#
# OUTPUT:
# ROC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_145625.xlsx
#
# PURPOSE:
# Finalize ROC qualitative evidence and organize the themes
# into evidence-based dimensions for questionnaire development.
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

TARGET = "ROC_FINAL_QUALITATIVE_ANALYSIS_20260825_145531"

possible_files = []

# First check the current working directory and common folders
search_locations = [
    Path("."),
    Path.home(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop",
]

for location in search_locations:
    if location.exists():
        for ext in [".xlsx", ".xlsm", ".xls"]:
            try:
                possible_files.extend(
                    location.rglob(TARGET + ext)
                )
            except Exception:
                pass

# Also check /mnt/data for uploaded/copied files
try:
    possible_files.extend(
        Path("/mnt/data").glob("*.xlsx")
    )
except Exception:
    pass

possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)

# Prefer an exact filename match
exact_files = [
    p for p in possible_files
    if p.stem == TARGET
]

if len(exact_files) > 0:
    INPUT_FILE = exact_files[0]

else:

    # If the original filename has been renamed by the upload system,
    # identify the workbook by its ROC-specific sheet.
    roc_candidates = []

    for p in possible_files:

        try:
            xl = pd.ExcelFile(
                p,
                engine="openpyxl"
            )

            if "11_Final_ROC_Evidence" in xl.sheet_names:
                roc_candidates.append(p)

        except Exception:
            pass

    if len(roc_candidates) > 0:
        INPUT_FILE = roc_candidates[0]

    else:
        raise FileNotFoundError(
            "\nROC input workbook could not be found.\n\n"
            "Expected filename:\n"
            f"{TARGET}.xlsx"
        )


print("=" * 80)
print("ROC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE)


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_ROC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


if final_sheet is None:

    raise ValueError(
        "11_Final_ROC_Evidence was not found."
    )


# ================================================================
# 6. READ FINAL ROC EVIDENCE
# ================================================================

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# ================================================================
# 7. FIND ROC THEME COLUMN
# ================================================================

theme_col = None

if "Final_ROC_Theme" in final_evidence.columns:

    theme_col = "Final_ROC_Theme"

else:

    for c in final_evidence.columns:

        if (
            "ROC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify the final ROC theme column."
    )


final_evidence[
    "Final_ROC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT COLUMNS
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_ROC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. PARTICIPANT COUNT
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


# Your qualitative dataset has 26 participants
if len(participants) > 0:

    n_participants = len(participants)

else:

    n_participants = 26


print(
    "\nParticipants used:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# ================================================================
# 12. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_evidence.columns:

    final_evidence = (
        final_evidence
        .sort_values(
            "Experts_Mentioning",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL ROC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_ROC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]

evidence_columns = [
    c for c in preferred_columns
    if c in final_evidence.columns
]

final_roc_evidence = final_evidence[
    evidence_columns
].copy()


# Prevent Rank duplication
if "Rank" in final_roc_evidence.columns:

    final_roc_evidence = (
        final_roc_evidence
        .drop(columns=["Rank"])
    )


final_roc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_roc_evidence) + 1
    )
)


# ================================================================
# 14. ROC DIMENSION MAPPING
# ================================================================
#
# These dimensions are intended for questionnaire-development
# purposes. They organize the final ROC themes according to
# what the experts describe as risk orchestration activities.
#
# They are NOT yet statistically validated subdimensions.
# ================================================================

def map_roc_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. RISK RESPONSE COORDINATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "coordination",
            "coordinating",
            "cross-functional",
            "cross functional",
            "interdepartmental",
            "between departments",
            "between functions",
            "stakeholder coordination",
            "supplier coordination",
            "partner coordination"

        ]
    ):

        return (
            "Risk Response Coordination"
        )


    # ------------------------------------------------------------
    # 2. RISK RESPONSE PRIORITIZATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "prioritization",
            "prioritisation",
            "priority",
            "prioritize",
            "prioritise",
            "resource allocation",
            "resource allocation decisions",
            "critical risks",
            "most critical"

        ]
    ):

        return (
            "Risk Response Prioritization"
        )


    # ------------------------------------------------------------
    # 3. RISK RESPONSE INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integrated response",
            "integrated action",
            "integrating",
            "integration",
            "combined response",
            "coordinated response",
            "holistic response",
            "response across"

        ]
    ):

        return (
            "Integrated Risk Response"
        )


    # ------------------------------------------------------------
    # 4. RESOURCE & ACTION ORCHESTRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "resource",
            "capacity",
            "reallocate",
            "reallocation",
            "deploy",
            "deployment",
            "mobilize",
            "mobilise",
            "action",
            "response action",
            "corrective action"

        ]
    ):

        return (
            "Resource & Action Orchestration"
        )


    # ------------------------------------------------------------
    # 5. ADAPTIVE RISK RESPONSE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "adapt",
            "adaptive",
            "adjust",
            "adjustment",
            "flexibility",
            "flexible response",
            "changing conditions",
            "changing circumstances",
            "dynamic response",
            "respond to changes"

        ]
    ):

        return (
            "Adaptive Risk Response"
        )


    # ------------------------------------------------------------
    # 6. RISK RESPONSE MONITORING & FOLLOW-UP
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "monitor response",
            "monitoring response",
            "follow-up",
            "follow up",
            "tracking",
            "response effectiveness",
            "response performance",
            "evaluate response",
            "review response"

        ]
    ):

        return (
            "Risk Response Monitoring & Follow-up"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_roc_evidence[
    "ROC_Dimension"
] = (
    final_roc_evidence[
        "Final_ROC_Theme"
    ]
    .apply(
        map_roc_dimension
    )
)


# ================================================================
# 15. ADDITIONAL MANUAL MAPPING
# ================================================================
#
# Only explicit labels are mapped here.
# Anything not confidently mapped remains Review Required.
# ================================================================

manual_mapping = {

    # Coordination
    "coordination":
        "Risk Response Coordination",

    "risk coordination":
        "Risk Response Coordination",

    "cross-functional coordination":
        "Risk Response Coordination",

    "cross functional coordination":
        "Risk Response Coordination",

    "stakeholder coordination":
        "Risk Response Coordination",

    "supplier coordination":
        "Risk Response Coordination",

    "partner coordination":
        "Risk Response Coordination",


    # Prioritization
    "prioritization":
        "Risk Response Prioritization",

    "prioritisation":
        "Risk Response Prioritization",

    "priority":
        "Risk Response Prioritization",

    "resource allocation":
        "Risk Response Prioritization",

    "critical risks":
        "Risk Response Prioritization",


    # Integration
    "integration":
        "Integrated Risk Response",

    "integrated response":
        "Integrated Risk Response",

    "integrated action":
        "Integrated Risk Response",

    "combined response":
        "Integrated Risk Response",

    "holistic response":
        "Integrated Risk Response",


    # Resource / action
    "resource":
        "Resource & Action Orchestration",

    "resource allocation decisions":
        "Resource & Action Orchestration",

    "reallocation":
        "Resource & Action Orchestration",

    "resource reallocation":
        "Resource & Action Orchestration",

    "deployment":
        "Resource & Action Orchestration",

    "mobilization":
        "Resource & Action Orchestration",

    "mobilisation":
        "Resource & Action Orchestration",


    # Adaptation
    "adaptation":
        "Adaptive Risk Response",

    "adaptive response":
        "Adaptive Risk Response",

    "flexibility":
        "Adaptive Risk Response",

    "flexible response":
        "Adaptive Risk Response",

    "dynamic response":
        "Adaptive Risk Response",


    # Monitoring
    "response monitoring":
        "Risk Response Monitoring & Follow-up",

    "monitoring response":
        "Risk Response Monitoring & Follow-up",

    "follow-up":
        "Risk Response Monitoring & Follow-up",

    "follow up":
        "Risk Response Monitoring & Follow-up",

    "response effectiveness":
        "Risk Response Monitoring & Follow-up",

    "response performance":
        "Risk Response Monitoring & Follow-up"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_roc_evidence[
            "Final_ROC_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_roc_evidence.loc[
        mask,
        "ROC_Dimension"
    ] = dimension


# ================================================================
# 16. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_roc_evidence
    .groupby(
        "ROC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_ROC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts_mentioning = 0


    # ------------------------------------------------------------
    # Use participant matrix when available
    # ------------------------------------------------------------

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()


        theme_candidates = [
            c
            for c in temp.columns
            if "theme" in str(c).lower()
        ]


        if len(theme_candidates) > 0:

            matrix_theme_col = (
                theme_candidates[0]
            )

        else:

            matrix_theme_col = (
                temp.columns[0]
            )


        temp["_Theme"] = (
            temp[
                matrix_theme_col
            ]
            .astype(str)
            .str.strip()
        )


        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        available_participants = [

            p

            for p in participants

            if p in matched.columns

        ]


        if (
            len(matched) > 0
            and len(available_participants) > 0
        ):

            vals = (
                matched[
                    available_participants
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce"
                )
                .fillna(0)
            )


            experts_mentioning = int(
                (
                    vals.sum(axis=0) > 0
                ).sum()
            )


    # ------------------------------------------------------------
    # Fallback
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            try:

                experts_mentioning = int(
                    pd.to_numeric(
                        group[
                            "Experts_Mentioning"
                        ],
                        errors="coerce"
                    )
                    .max()
                )

            except:

                experts_mentioning = 0


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "ROC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_ROC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(columns=["Rank"])
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "ROC_Dimension",

    "Final_ROC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_roc_evidence.columns
]


dimension_themes = final_roc_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "ROC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_roc_evidence[
    final_roc_evidence[
        "ROC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_ROC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()


    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All ROC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. CODING DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c
        for c in coding_audit.columns
        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = (
            decision_columns[0]
        )


        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All ROC coding decisions are present."
                ]

            })

        else:

            decision_check = missing.copy()


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final ROC evidence available",

    "Result":
        "PASS"
        if len(final_roc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_roc_evidence)} final ROC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "ROC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative evidence-based dimensions"

})


review_count = (

    len(unmapped_check)

    if "Final_ROC_Theme"
    in unmapped_check.columns

    else 0

)


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        review_count,

    "Details":
        "Themes assigned to Review Required"

})


duplicate_count = int(
    final_roc_evidence[
        "Final_ROC_Theme"
    ]
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        duplicate_count,

    "Details":
        "Case-insensitive duplicate labels"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:
        return sheets[sheet].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"ROC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    final_roc_evidence.to_excel(
        writer,
        sheet_name="13_Final_ROC_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="14_ROC_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="15_ROC_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("ROC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)


print("\nNumber of final ROC themes:")
print(
    len(final_roc_evidence)
)


print("\nNumber of ROC dimensions:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 80)
print("ROC DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("QUALITY CHECKS")
print("-" * 80)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("IMPORTANT")
print("=" * 80)

print("""
The two sheets you should use for the next questionnaire stage are:

14_ROC_Dimension_Summary
15_ROC_Dimension_Themes

They connect your qualitative evidence to questionnaire-item
development.

The dimensions are NOT automatically treated as separate
statistical constructs. They are qualitative evidence-based
groupings that help us determine what aspects of ROC should
be represented in the questionnaire.

The next stage is to convert the strongest, non-overlapping
ROC themes into candidate reflective questionnaire items.
""")

ROC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\ROC_FINAL_QUALITATIVE_ANALYSIS_20260825_145531.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_ROC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Participants used: 26


ROC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\ROC_FINAL_EVIDENCE_AND_DIMENSIONS_20260825_145625.xlsx

Number of final ROC themes:
62

Number of ROC dimensions:
6


--------------------------------------------------------------------------------


# Risk Signal Quality (RSQ)

In [ ]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ================================================================
# RSQ CODING
# INPUT: Themes.xlsx
# OUTPUT: RSQ_Coding_20260825_134601.xlsx
# ================================================================

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)

# ================================================================
# 1. FIND PARTICIPANT SHEETS
# ================================================================

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ================================================================
# 2. EXTRACT RSQ RESPONSES
# ================================================================

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        # Find RSQ regardless of capitalization
        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "RSQ":

                construct_position = position
                break

        if construct_position is None:
            continue

        # Everything after RSQ is response/theme information
        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "RSQ",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ================================================================
# 3. CHECK RSQ RESPONSES
# ================================================================

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No RSQ responses were found.

        Check that the construct is written as RSQ
        somewhere in the participant sheets.
        """
    )

print("\nRSQ original responses:")
print(len(original_df))


# ================================================================
# 4. SPLIT INTO RAW THEMES
# ================================================================

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    # Standardize separators
    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "RSQ",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ================================================================
# 5. CLEAN THEMES
# ================================================================

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

# Remove empty
clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

# Remove duplicate theme within participant
clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nRSQ cleaned theme observations:")
print(len(clean_df))


# ================================================================
# 6. DISPLAY ALL UNIQUE RSQ RAW THEMES
# ================================================================

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(drop=True)
)

print("\n==============================")
print("RSQ RAW THEMES")
print("==============================")

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ================================================================
# 7. RSQ NORMALIZATION DICTIONARY
# ================================================================
#
# IMPORTANT:
# Do NOT invent conceptual mergers before seeing your actual
# RSQ themes.
#
# First, exact duplicates / capitalization variants are handled
# automatically.
#
# Add conceptual mappings below AFTER reviewing the raw-theme list.
#
# Format:
#
# "raw theme": "Normalized Theme"
#
# ================================================================

RSQ_NORMALIZATION = {

    # ------------------------------------------------------------
    # ADD YOUR RSQ NORMALIZATION RULES HERE
    # ------------------------------------------------------------
    
    # Example only:
    #
    # "rapid sensing": "Rapid risk sensing",
    # "fast risk sensing": "Rapid risk sensing",
    # "early risk identification": "Early risk identification",

}


# ================================================================
# 8. APPLY NORMALIZATION
# ================================================================

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    # ------------------------------------------------------------
    # If researcher has supplied a mapping
    # ------------------------------------------------------------

    if theme_key in RSQ_NORMALIZATION:

        normalized = RSQ_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        # --------------------------------------------------------
        # If no mapping supplied, keep original theme temporarily
        # --------------------------------------------------------

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "No conceptual merger specified; "
            "retained pending researcher review."
        )


# ================================================================
# 9. MAP NORMALIZED THEMES BACK TO PARTICIPANTS
# ================================================================

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ================================================================
# 10. PARTICIPANT × RSQ THEME MATRIX
# ================================================================

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source[
        "Normalized_Theme"
    ],
    matrix_source[
        "Participant"
    ]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    /
    len(participants)
    *
    100
).round(1)


# ================================================================
# 11. RSQ THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ================================================================
# 12. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Count"
    )
)


# ================================================================
# 13. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    coded_df
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "RSQ_Theme_Count"
]


# ================================================================
# 14. SAVE EXCEL
# ================================================================

output_file = Path(
    "RSQ_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_RSQu",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ================================================================
# 15. FINAL OUTPUT
# ================================================================

print("\n")
print("=" * 60)
print("RSQ CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique RSQ raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized RSQ themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")
print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")
print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

RSQ original responses:
26

RSQ cleaned theme observations:
106

RSQ RAW THEMES
               Raw_Theme              Theme_Clean                Theme_Key
                Accuracy                 Accuracy                 accuracy
       action assessment        action assessment        action assessment
           actionability            actionability            actionability
  Anomaly discrimination   Anomaly discrimination   anomaly discrimination
attention prioritization attention prioritization attention prioritization
        Change detection         Change detection         change detection
                 clarity                  clarity                  clarity
                 Clarity                  

In [ ]:
# ================================================================
# COMPLETE RSQ QUALITATIVE ANALYSIS
# ================================================================
#
# INPUT:
#     RSQ_Coding_20260825_134601.xlsx
#
# OUTPUT:
#     RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134708.xlsx
#
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

input_file = Path("RSQ_Coding_20260825_134601.xlsx")

if not input_file.exists():

    possible_files = list(Path(".").glob("*RSQ*.xlsx"))

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "\nNo RSQ Excel file was found.\n"
            "Please put RSQ_Complete_Qualitative_Coding.xlsx "
            "in the same folder as your notebook."
        )

    elif len(possible_files) == 1:
        input_file = possible_files[0]

    else:
        print("Multiple RSQ files were found:")
        for i, f in enumerate(possible_files):
            print(f"{i}: {f.name}")

        raise ValueError(
            "\nMore than one RSQ Excel file was found. "
            "Keep only the correct completed RSQ workbook."
        )


print("=" * 70)
print("RSQ QUALITATIVE ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for sheet in excel.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. FIND REQUIRED SHEETS
# ================================================================

def find_sheet(possible_names):

    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # More flexible matching
    for sheet in excel.sheet_names:

        clean_sheet = (
            sheet.lower()
            .replace(" ", "")
            .replace("_", "")
        )

        for name in possible_names:

            clean_name = (
                name.lower()
                .replace(" ", "")
                .replace("_", "")
            )

            if clean_name in clean_sheet:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_RSQ",
    "01_Original",
    "Original_RSQ",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_RSQ",
    "02_Raw",
    "Raw_RSQ",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_RSQ",
    "03_Cleaned",
    "Cleaned_RSQ",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")

print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


if any(
    x is None
    for x in [
        original_sheet,
        raw_sheet,
        clean_sheet,
        coding_sheet
    ]
):

    raise ValueError(
        "\nCould not identify all required sheets.\n"
        "Check the sheet names printed above."
    )


# ================================================================
# 4. READ DATA
# ================================================================

original_df = pd.read_excel(
    input_file,
    sheet_name=original_sheet
)

raw_df = pd.read_excel(
    input_file,
    sheet_name=raw_sheet
)

clean_df = pd.read_excel(
    input_file,
    sheet_name=clean_sheet
)

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]


# ================================================================
# 6. CHECK REQUIRED CODING COLUMNS
# ================================================================

required_columns = [
    "Theme_Key",
    "Raw_Theme",
    "Normalized_Theme",
    "Decision"
]

missing = [
    c
    for c in required_columns
    if c not in coding_df.columns
]

if missing:

    raise ValueError(
        "\nThe following columns are missing from the "
        "Normalized Coding sheet:\n"
        + str(missing)
        + "\n\nAvailable columns are:\n"
        + str(list(coding_df.columns))
    )


# ================================================================
# 7. IDENTIFY PARTICIPANT COLUMN
# ================================================================

participant_candidates = [
    "Participant",
    "participant",
    "Participant_ID",
    "Participant ID"
]

participant_column = None

for c in participant_candidates:

    if c in clean_df.columns:
        participant_column = c
        break

if participant_column is None:

    raise ValueError(
        "\nParticipant column was not found in the "
        "Cleaned RSQ sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


# Standardize to Participant

if participant_column != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_column:
            "Participant"
        }
    )


# ================================================================
# 8. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", ""],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 9. CLEAN CLEANED-THEME DATA
# ================================================================

if "Theme_Key" not in clean_df.columns:

    raise ValueError(
        "\nTheme_Key is missing from the Cleaned RSQ sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ================================================================
# 10. IDENTIFY PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nParticipants identified:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 11. MERGE CLEANED THEMES WITH CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 12. CHECK UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()


if len(missing_mapping) > 0:

    print("\nWARNING:")
    print(
        "Unmapped RSQ themes:",
        len(missing_mapping)
    )

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll RSQ cleaned themes have normalized mappings."
    )


# ================================================================
# 13. PARTICIPANT × NORMALIZED THEME DATA
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


# ================================================================
# 14. CREATE PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# Try to preserve P01-P26 ordering

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing_participants = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other_participants = [
    p
    for p in matrix.columns
    if p not in existing_participants
]

matrix = matrix.reindex(
    columns=existing_participants + other_participants,
    fill_value=0
)


matrix = matrix.reset_index()


# ================================================================
# 15. CALCULATE FREQUENCY
# ================================================================

matrix["Frequency"] = matrix[
    existing_participants + other_participants
].sum(axis=1)


total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 16. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    by=[
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 17. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_RSQ_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 18. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(prevalence_category)
)


# ================================================================
# 19. CODING AUDIT
# ================================================================

coding_audit = coding_df.copy()


theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_audit.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)


coding_audit["Percentage_of_Experts"] = (
    coding_audit["Experts_Mentioning"]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 20. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary["Percentage"] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 21. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 22. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 23. CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["RSQ"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 24. FINAL RSQ EVIDENCE TABLE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_RSQ_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 25. QUALITY CHECKS
# ================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme combinations",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        (
            len(coded_df)
            -
            len(participant_theme)
        ),

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants >= 1
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"
    ]
})


# ================================================================
# 26. SAVE OUTPUT
# ================================================================

output_file = Path(
    "RSQ_FINAL_QUALITATIVE_ANALYSIS_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RSQ_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 27. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("RSQ ANALYSIS COMPLETED")
print("=" * 70)

print("\nParticipants:", total_participants)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized RSQ themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"]
        == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"]
        == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 28. DISPLAY FINAL RSQ THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL RSQ THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_RSQ_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

print(
    "\nComplete RSQ qualitative analysis workbook created successfully."
)

RSQ QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\RSQ_Coding_20260825_134601.xlsx

Sheets found:
 - 01_Original_RSQu
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_RSQu
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Participants identified:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

All RSQ cleaned themes have normalized mappings.


RSQ ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 106
Cleaned theme observations: 106
Unique raw themes: 53
Final normalized RSQ themes

In [ ]:
# ================================================================
# RSQ — COMPLETE QUALITATIVE CODING WORKFLOW
# ================================================================
# INPUT:
#   RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134708.xlsx
#
# OUTPUT:
#   RSQ_COMPLETE_QUALITATIVE_CODING_20260825_142706.xlsx
#
# Creates:
#   01_Original_RSQ
#   02_Raw_Themes
#   03_Cleaned_Themes
#   04_Coding_Audit
#   05_Participant_Matrix
#   06_Theme_Summary
#   07_Raw_Theme_Summary
#   08_Decision_Summary
#   09_Participant_Coverage
#   10_Unmapped_Check
#   11_Decision_Check
#   12_Quality_Checks
#   13_Final_RSQ_Evidence
#   14_RSQ_Dimension_Summary
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

target_name = "RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134708"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").glob(target_name + ext)
    )

if len(possible_files) == 0:

    # Also search current directory recursively
    for ext in [".xlsx", ".xlsm", ".xls"]:
        possible_files.extend(
            Path(".").rglob(target_name + ext)
        )

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\nCould not find:\n"
        f"{target_name}.xlsx\n\n"
        "Put the RSQ Excel file in the same folder as "
        "your Python/Jupyter working directory."
    )

INPUT_FILE = possible_files[0]

print("=" * 75)
print("RSQ COMPLETE QUALITATIVE CODING")
print("=" * 75)

print("\nInput file found:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ EXCEL WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. HELPER FUNCTIONS
# ================================================================

def normalize_col_name(x):

    x = str(x).strip()

    x = x.replace(" ", "_")
    x = x.replace("-", "_")

    return x


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def find_column(df, candidates):

    normalized = {
        normalize_col_name(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        key = normalize_col_name(candidate).lower()

        if key in normalized:

            return normalized[key]

    # Fuzzy fallback
    for col in df.columns:

        col_clean = normalize_col_name(col).lower()

        for candidate in candidates:

            candidate_clean = (
                normalize_col_name(candidate)
                .lower()
            )

            if (
                candidate_clean in col_clean
                or col_clean in candidate_clean
            ):

                return col

    return None


def safe_sheet_name(name):

    # Excel sheet names cannot exceed 31 characters
    return name[:31]


# ================================================================
# 4. FIND RAW THEME SHEET
# ================================================================

raw_sheet = None

for s in xls.sheet_names:

    sl = s.lower()

    if "raw" in sl and "theme" in sl:
        raw_sheet = s
        break


if raw_sheet is None:

    # fallback
    for s in xls.sheet_names:

        if "raw" in s.lower():
            raw_sheet = s
            break


if raw_sheet is None:

    raise ValueError(
        "Could not find the Raw Themes sheet."
    )


print("\nRaw theme sheet:")
print(raw_sheet)


# ================================================================
# 5. READ RAW THEMES
# ================================================================

raw = pd.read_excel(
    INPUT_FILE,
    sheet_name=raw_sheet
)

raw.columns = [
    normalize_col_name(c)
    for c in raw.columns
]


print("\nRaw theme columns:")
print(list(raw.columns))


# ================================================================
# 6. IDENTIFY REQUIRED COLUMNS
# ================================================================

participant_col = find_column(
    raw,
    [
        "Participant",
        "Participant_ID",
        "ParticipantID",
        "Expert",
        "Expert_ID"
    ]
)

construct_col = find_column(
    raw,
    [
        "Construct",
        "Construct_Name"
    ]
)

theme_col = find_column(
    raw,
    [
        "Raw_Theme",
        "RawTheme",
        "Theme",
        "Raw_Theme_Name"
    ]
)


if participant_col is None:

    raise ValueError(
        "\nParticipant column not found.\n"
        f"Available columns: {list(raw.columns)}"
    )


if construct_col is None:

    raise ValueError(
        "\nConstruct column not found.\n"
        f"Available columns: {list(raw.columns)}"
    )


if theme_col is None:

    raise ValueError(
        "\nRaw Theme column not found.\n"
        f"Available columns: {list(raw.columns)}"
    )


# ================================================================
# 7. EXTRACT RSQ
# ================================================================

raw["__Construct_Clean"] = (
    raw[construct_col]
    .astype(str)
    .str.strip()
    .str.upper()
)

rsq = raw[
    raw["__Construct_Clean"] == "RSQ"
].copy()


if len(rsq) == 0:

    raise ValueError(
        "\nNo RSQ rows were found.\n\n"
        "Check the Construct column and confirm "
        "that RSQ is used as the construct name."
    )


rsq = rsq.rename(
    columns={
        participant_col: "Participant",
        construct_col: "Construct",
        theme_col: "Raw_Theme"
    }
)


rsq["Participant"] = (
    rsq["Participant"]
    .apply(clean_text)
)


rsq["Construct"] = "RSQ"


rsq["Raw_Theme"] = (
    rsq["Raw_Theme"]
    .apply(clean_text)
)


# Remove empty themes
rsq = rsq[
    rsq["Raw_Theme"] != ""
].copy()


# ================================================================
# 8. STANDARDIZE PARTICIPANT IDS
# ================================================================

def standardize_participant(x):

    x = str(x).strip()

    match = re.search(
        r"(\d+)",
        x
    )

    if match:

        number = int(match.group(1))

        return f"P{number:02d}"

    return x


rsq["Participant"] = (
    rsq["Participant"]
    .apply(standardize_participant)
)


# ================================================================
# 9. CREATE THEME KEY
# ================================================================

rsq["Theme_Key"] = (

    rsq["Raw_Theme"]

    .str.lower()

    .str.strip()

)


# ================================================================
# 10. FIND CODING AUDIT
# ================================================================

audit_sheet = None

for s in xls.sheet_names:

    sl = s.lower()

    if "audit" in sl:

        audit_sheet = s
        break


if audit_sheet is None:

    raise ValueError(
        "Could not find the Coding Audit sheet."
    )


print("\nCoding audit sheet:")
print(audit_sheet)


audit = pd.read_excel(
    INPUT_FILE,
    sheet_name=audit_sheet
)


audit.columns = [
    normalize_col_name(c)
    for c in audit.columns
]


print("\nCoding audit columns:")
print(list(audit.columns))


# ================================================================
# 11. FIND AUDIT COLUMNS
# ================================================================

audit_raw = find_column(
    audit,
    [
        "Raw_Theme",
        "RawTheme",
        "Theme"
    ]
)

audit_normalized = find_column(
    audit,
    [
        "Normalized_Theme",
        "NormalizedTheme",
        "Normalized_Theme_Name",
        "Normalized Theme"
    ]
)

audit_decision = find_column(
    audit,
    [
        "Decision",
        "Coding_Decision"
    ]
)

audit_reason = find_column(
    audit,
    [
        "Reason",
        "Decision_Reason",
        "Rationale"
    ]
)


if audit_raw is None:

    raise ValueError(
        "Raw Theme column not found in Coding Audit."
    )


if audit_normalized is None:

    raise ValueError(
        "Normalized Theme column not found in Coding Audit."
    )


if audit_decision is None:

    raise ValueError(
        "Decision column not found in Coding Audit."
    )


# ================================================================
# 12. STANDARDIZE AUDIT
# ================================================================

audit2 = pd.DataFrame()

audit2["Raw_Theme"] = (
    audit[audit_raw]
    .apply(clean_text)
)

audit2["Theme_Key"] = (
    audit2["Raw_Theme"]
    .str.lower()
    .str.strip()
)

audit2["Normalized_Theme"] = (
    audit[audit_normalized]
    .apply(clean_text)
)

audit2["Decision"] = (
    audit[audit_decision]
    .apply(clean_text)
    .str.title()
)

if audit_reason is not None:

    audit2["Reason"] = (
        audit[audit_reason]
        .apply(clean_text)
    )

else:

    audit2["Reason"] = ""


# ================================================================
# 13. REMOVE DUPLICATE AUDIT ENTRIES
# ================================================================

audit2 = (
    audit2
    .drop_duplicates(
        subset=["Theme_Key"],
        keep="last"
    )
)


# ================================================================
# 14. MERGE CODING AUDIT WITH RSQ
# ================================================================

coded = rsq.merge(

    audit2[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision",
            "Reason"
        ]
    ],

    on="Theme_Key",

    how="left"

)


# ================================================================
# 15. CHECK UNMAPPED THEMES
# ================================================================

coded["Normalized_Theme"] = (
    coded["Normalized_Theme"]
    .apply(clean_text)
)


coded["Decision"] = (
    coded["Decision"]
    .apply(clean_text)
)


unmapped = coded[
    coded["Normalized_Theme"] == ""
].copy()


missing_decision = coded[
    coded["Decision"] == ""
].copy()


print("\n")
print("-" * 75)
print("CODING CHECK")
print("-" * 75)

print(
    "Total RSQ raw-theme observations:",
    len(coded)
)

print(
    "Unmapped observations:",
    len(unmapped)
)

print(
    "Missing decisions:",
    len(missing_decision)
)


# ================================================================
# 16. CREATE CLEANED THEMES
# ================================================================

cleaned_themes = coded[
    [
        "Participant",
        "Construct",
        "Raw_Theme",
        "Theme_Key"
    ]
].copy()


# ================================================================
# 17. CODING AUDIT
# ================================================================

coding_audit = coded[
    [
        "Participant",
        "Construct",
        "Raw_Theme",
        "Theme_Key",
        "Normalized_Theme",
        "Decision",
        "Reason"
    ]
].copy()


# ================================================================
# 18. PARTICIPANT LIST
# ================================================================

participants_detected = sorted(
    coded["Participant"]
    .dropna()
    .unique()
)


# ================================================================
# 19. PARTICIPANT × NORMALIZED THEME MATRIX
# ================================================================

valid_coded = coded[
    coded["Normalized_Theme"] != ""
].copy()


# Remove duplicate mention of the same normalized theme
# by the same participant

valid_coded_unique = (
    valid_coded
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)


participant_matrix = pd.crosstab(

    valid_coded_unique[
        "Normalized_Theme"
    ],

    valid_coded_unique[
        "Participant"
    ]

)


participant_matrix = (
    participant_matrix
    .reindex(
        columns=participants_detected,
        fill_value=0
    )
)


# ================================================================
# 20. FREQUENCY AND PREVALENCE
# ================================================================

participant_matrix[
    "Experts_Mentioning"
] = (
    participant_matrix[
        participants_detected
    ]
    .sum(axis=1)
)


n_participants = len(
    participants_detected
)


if n_participants > 0:

    participant_matrix[
        "Expert_Prevalence_%"
    ] = (

        participant_matrix[
            "Experts_Mentioning"
        ]

        /

        n_participants

        *

        100

    ).round(1)

else:

    participant_matrix[
        "Expert_Prevalence_%"
    ] = 0


# ================================================================
# 21. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(x):

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


participant_matrix[
    "Prevalence_Category"
] = (
    participant_matrix[
        "Expert_Prevalence_%"
    ]
    .apply(prevalence_category)
)


# ================================================================
# 22. THEME SUMMARY
# ================================================================

theme_summary = (
    participant_matrix
    .reset_index()
)


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "RSQ_Theme"
    }
)


theme_summary = theme_summary.sort_values(
    [
        "Experts_Mentioning",
        "RSQ_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


theme_summary.insert(
    0,
    "Rank",
    range(
        1,
        len(theme_summary) + 1
    )
)


# ================================================================
# 23. RAW THEME SUMMARY
# ================================================================

raw_theme_summary = (

    coded

    .groupby(
        [
            "Raw_Theme",
            "Normalized_Theme",
            "Decision"
        ],
        dropna=False
    )

    .agg(
        Experts_Mentioning=(
            "Participant",
            "nunique"
        )
    )

    .reset_index()

)


raw_theme_summary[
    "Expert_Prevalence_%"
] = (

    raw_theme_summary[
        "Experts_Mentioning"
    ]

    /

    n_participants

    *

    100

).round(1)


# ================================================================
# 24. DECISION SUMMARY
# ================================================================

decision_summary = (

    coded

    .groupby(
        [
            "Normalized_Theme",
            "Decision"
        ],
        dropna=False
    )

    .agg(
        Number_of_Raw_Themes=(
            "Raw_Theme",
            "nunique"
        )
    )

    .reset_index()

)


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (

    valid_coded_unique

    .groupby(
        "Participant"
    )

    ["Normalized_Theme"]

    .nunique()

    .reset_index()

)


participant_coverage = (
    participant_coverage
    .rename(
        columns={
            "Normalized_Theme":
                "Number_of_Normalized_Themes"
        }
    )
)


number_normalized_themes = (
    valid_coded_unique[
        "Normalized_Theme"
    ].nunique()
)


if number_normalized_themes > 0:

    participant_coverage[
        "Coverage_Percentage"
    ] = (

        participant_coverage[
            "Number_of_Normalized_Themes"
        ]

        /

        number_normalized_themes

        *

        100

    ).round(1)

else:

    participant_coverage[
        "Coverage_Percentage"
    ] = 0


# ================================================================
# 26. FINAL RSQ EVIDENCE
# ================================================================

final_rsq_evidence = theme_summary.copy()


final_rsq_evidence = final_rsq_evidence[
    [
        "Rank",
        "RSQ_Theme",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
]


# ================================================================
# 27. RSQ DIMENSION MAPPING
# ================================================================
#
# IMPORTANT:
# These dimensions follow the RSQ construct definition:
#
# Timeliness
# Accuracy
# Reliability/Credibility
# Clarity/Interpretability
# Actionability
#
# The code attempts to map normalized themes using their wording.
# Themes that cannot be confidently mapped are marked:
# "Review Required"
#
# This prevents the code from silently inventing dimensions.
# ================================================================

def map_rsq_dimension(theme):

    t = str(theme).lower()

    # ------------------------------------------------------------
    # TIMELINESS
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "timely",
            "timeliness",
            "real-time",
            "realtime",
            "speed",
            "rapid",
            "quick",
            "immediate",
            "early",
            "prompt",
            "up-to-date",
            "current"
        ]
    ):
        return "Timeliness"


    # ------------------------------------------------------------
    # ACCURACY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "accuracy",
            "accurate",
            "correct",
            "precision",
            "error-free",
            "validity",
            "valid"
        ]
    ):
        return "Accuracy"


    # ------------------------------------------------------------
    # RELIABILITY / CREDIBILITY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "reliab",
            "trust",
            "credible",
            "credibility",
            "consistent",
            "consistency",
            "integrity",
            "verified",
            "verification",
            "authentic"
        ]
    ):
        return "Reliability / Credibility"


    # ------------------------------------------------------------
    # CLARITY / INTERPRETABILITY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "clear",
            "clarity",
            "understand",
            "interpret",
            "interpretability",
            "transparent",
            "transparency",
            "visibility"
        ]
    ):
        return "Clarity / Interpretability"


    # ------------------------------------------------------------
    # ACTIONABILITY
    # ------------------------------------------------------------

    if any(
        k in t
        for k in [
            "action",
            "actionable",
            "decision",
            "response",
            "respond",
            "useful",
            "usable",
            "operational",
            "support",
            "warning"
        ]
    ):
        return "Actionability"


    # ------------------------------------------------------------
    # UNRESOLVED
    # ------------------------------------------------------------

    return "Review Required"


theme_summary[
    "RSQ_Dimension"
] = (
    theme_summary[
        "RSQ_Theme"
    ]
    .apply(map_rsq_dimension)
)


# ================================================================
# 28. DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    theme_summary
    .groupby("RSQ_Dimension")
):

    included_themes = (
        group[
            "RSQ_Theme"
        ]
        .astype(str)
        .tolist()
    )


    experts = int(
        group[
            "Experts_Mentioning"
        ].max()
    )


    # More robust dimension-level prevalence:
    # calculate unique participants mentioning ANY theme
    # belonging to the dimension.

    dimension_themes = set(
        included_themes
    )


    dimension_data = valid_coded_unique[
        valid_coded_unique[
            "Normalized_Theme"
        ].isin(
            dimension_themes
        )
    ]


    dimension_experts = (
        dimension_data[
            "Participant"
        ]
        .nunique()
    )


    dimension_prevalence = (

        dimension_experts

        /

        n_participants

        *

        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "RSQ_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(included_themes),

        "Experts_Mentioning":
            dimension_experts,

        "Expert_Prevalence_%":
            round(
                dimension_prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                dimension_prevalence
            ),

        "Included_RSQ_Themes":
            "; ".join(
                included_themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 29. UNMAPPED CHECK
# ================================================================

if len(unmapped) > 0:

    unmapped_check = unmapped[
        [
            "Participant",
            "Raw_Theme",
            "Theme_Key"
        ]
    ].drop_duplicates()

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All RSQ raw themes were mapped."
        ]

    })


# ================================================================
# 30. DECISION CHECK
# ================================================================

if len(missing_decision) > 0:

    decision_check = missing_decision[
        [
            "Participant",
            "Raw_Theme",
            "Theme_Key",
            "Normalized_Theme"
        ]
    ].drop_duplicates()

else:

    decision_check = pd.DataFrame({

        "Status": [
            "PASS — All RSQ themes have a coding decision."
        ]

    })


# ================================================================
# 31. QUALITY CHECKS
# ================================================================

quality_checks = pd.DataFrame({

    "Quality_Check": [

        "Input file found",

        "RSQ observations extracted",

        "Participants detected",

        "Raw themes identified",

        "Normalized themes identified",

        "Unmapped themes",

        "Missing decisions",

        "Participant matrix created",

        "Dimension mapping completed"

    ],

    "Result": [

        "PASS",

        len(coded),

        n_participants,

        coded[
            "Raw_Theme"
        ].nunique(),

        coded[
            "Normalized_Theme"
        ].replace(
            "",
            np.nan
        )
        .nunique(),

        len(unmapped),

        len(missing_decision),

        "PASS",

        len(
            dimension_summary
        )

    ],

    "Status": [

        "OK",

        "OK" if len(coded) > 0 else "CHECK",

        "OK" if n_participants > 0 else "CHECK",

        "OK",

        "OK",

        "PASS" if len(unmapped) == 0 else "CHECK",

        "PASS" if len(missing_decision) == 0 else "CHECK",

        "OK",

        "OK"

    ]

})


# ================================================================
# 32. OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

OUTPUT_FILE = Path(
    f"RSQ_COMPLETE_QUALITATIVE_CODING_{timestamp}.xlsx"
)


# ================================================================
# 33. WRITE EXCEL
# ================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    # ------------------------------------------------------------
    # 01
    # ------------------------------------------------------------

    rsq.to_excel(
        writer,
        sheet_name="01_Original_RSQ",
        index=False
    )


    # ------------------------------------------------------------
    # 02
    # ------------------------------------------------------------

    rsq[
        [
            "Participant",
            "Construct",
            "Raw_Theme",
            "Theme_Key"
        ]
    ].to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 03
    # ------------------------------------------------------------

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 04
    # ------------------------------------------------------------

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    # ------------------------------------------------------------
    # 05
    # ------------------------------------------------------------

    participant_matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix"
    )


    # ------------------------------------------------------------
    # 06
    # ------------------------------------------------------------

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 07
    # ------------------------------------------------------------

    raw_theme_summary.to_excel(
        writer,
        sheet_name="07_Raw_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 08
    # ------------------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 09
    # ------------------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------------------
    # 10
    # ------------------------------------------------------------

    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 11
    # ------------------------------------------------------------

    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 12
    # ------------------------------------------------------------

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # 13
    # ------------------------------------------------------------

    final_rsq_evidence.to_excel(
        writer,
        sheet_name="13_Final_RSQ_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # 14
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_RSQ_Dimension_Summary",
        index=False
    )


# ================================================================
# 34. FINAL REPORT
# ================================================================

print("\n")
print("=" * 75)
print("RSQ CODING COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "\nInput:",
    INPUT_FILE.name
)

print(
    "Output:",
    OUTPUT_FILE.resolve()
)

print("\nRSQ statistics:")

print(
    "Raw observations:",
    len(coded)
)

print(
    "Unique raw themes:",
    coded["Raw_Theme"].nunique()
)

print(
    "Normalized themes:",
    coded[
        "Normalized_Theme"
    ]
    .replace(
        "",
        np.nan
    )
    .nunique()
)

print(
    "Participants:",
    n_participants
)

print(
    "Unmapped themes:",
    len(unmapped)
)

print(
    "Missing decisions:",
    len(missing_decision)
)

print(
    "RSQ dimensions:",
    len(dimension_summary)
)


print("\n")
print("=" * 75)
print("RSQ DIMENSION SUMMARY")
print("=" * 75)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )


print("\n")
print("=" * 75)
print("OUTPUT CREATED")
print("=" * 75)

print(
    OUTPUT_FILE.resolve()
)

print("\nYou can now open the output Excel workbook.")

RSQ COMPLETE QUALITATIVE CODING

Input file found:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Digital_Capabilities_Risk_Intelligence_Orchestration_Resilience_Model_4\RSQ_FINAL_QUALITATIVE_ANALYSIS_20260825_134708.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_RSQ_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Raw theme sheet:
02_Raw_Themes

Raw theme columns:
['Participant', 'Construct', 'Raw_Theme', 'Source_Sheet', 'Source_Row']

Coding audit sheet:
04_Coding_Audit

Coding audit columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason', 'Experts_Mentioning', 'Percentage_of_Experts']


---------------------------------------------------------------------------
CODING CHECK
----------------